## Defining Variabels

In [0]:
catalog = "akibit_catalog"
schema = "akibit_schema"
volume = "akibit_volume"
download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
file_name = "rows.csv"
table_name = "akibit_table"
path_volume = "/Volumes/"+ catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema
print("path_volume = " f'{path_volume}')
print("path_table = " f'{path_table}')
print("table_name = " f'{table_name}')
print("file_name = " f'{file_name}')
print("download_url = " f'{download_url}')



## Loading csv file from url

In [0]:
dbutils.fs.cp(f"{download_url}", f"{path_volume}/{file_name}")

unfortunately, this website is blocked so I need to download data manualy and add to the volume

## Create a Test DataFrame

In [0]:
data = [[2021, "test", "Albany", "M", 42]]
columns = ["Year", "First_Name", "County", "Sex", "Count"]
df1 = spark.createDataFrame(data, schema = "Year int, First_Name STRING, County STRING, Sex STRING, Count INT")
display(df1)


## Load data into a DataFrame from csv file

In [0]:
df_csv = spark.read.csv(f"{path_volume}/{file_name}", header=True, inferSchema=True, sep=",")
display(df_csv)


In [0]:
df_csv.show()

## View and interact with Data Frame

In [0]:
print("df_csv schema:")
df_csv.printSchema()
print("df1 schema:")
df1.printSchema()

## Other Data Frame Operations

In [0]:
df_csv.count()


In [0]:
df_csv.columns

In [0]:
df_csv.describe().show()

In [0]:
dir(df_csv)

In [0]:
display(df_csv.summary())

In [0]:
df_csv.summary()

## Combine DataFrames

In [0]:
df = df1.union(df_csv)
display(df)
df.printSchema()

## Filtering DataFrame

In [0]:
# method 1: by filter method

df.filter(df["Count"]>50).orderBy(df["Count"].desc()).show()

In [0]:
# method 2: by "where" method

display(df.where(df["Count"]>50).orderBy(df["Count"].desc()))

# Select columns from a DAtaFram and order by frequency

In [0]:
from pyspark.sql.functions import desc

display(df.select("First_Name", "Count").orderBy(desc("Count")))

## Create a subset DataFrame


In [0]:
subsetDF = df.filter((df["Year"] == 2015) & (df["Sex"] == "F")).select("First_Name", "Count")
display(subsetDF)


## Rename a column


In [0]:
subsetDF = subsetDF.withColumnRenamed("Count", "Total")
display(subsetDF)

## Save the DataFrame

In [0]:
print(path_table)

In [0]:
print(table_name)

In [0]:
print("saveAsTable =" f'{path_table}.{table_name}')

In [0]:
# Save the dataframe into a table
df.write.mode("overwrite").saveAsTable(f"{path_table}.{table_name}")
                                    

In [0]:
df.write.mode("overwrite").saveAsTable("akibit_catalog.akibit_schema.akibit_table")

In [0]:
## Safe DataFrame to json_filse
df.write.format("json").mode("overwrite").save("/Volumes/akibit_catalog/akibit_schema/akibit_volume/json_data")
## Read DataFrame from json_file
df = spark.read.format("json").load("/Volumes/akibit_catalog/akibit_schema/akibit_volume/json_data")
display(df)


In [0]:
display(spark.sql(f"Select * from {path_table}.{table_name}"))

In [0]:
%pwd

In [0]:
# 1) W jakim jestem KATALOGU i SCHEMACIE? (kontekst SQL)
print("Katalog:", spark.catalog.currentCatalog())
print("Schemat:", spark.catalog.currentDatabase())

# 2) Gdzie „jestem" jako notebook? (katalog roboczy — to co zwraca %pwd)
import os
print("Katalog roboczy:", os.getcwd())

# 3) Co jest w moim volumenie?
print(dbutils.fs.ls("/Volumes/akibit_catalog/akibit_schema/akibit_volume"))

In [0]:

display(spark.sql("select * from json_data"))


In [0]:
%sql
explain(select * from akibit_catalog.akibit_schema.akibit_table where First_Name !="")

In [0]:
df = spark.read.json("/Volumes/akibit_catalog/akibit_schema/akibit_volume/json_data")
df.explain()